<a href="https://colab.research.google.com/github/lindaikponmwen/Machine-Learning/blob/main/PrivacyPreserving_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from sklearn.model_selection import train_test_split  # For splitting the data
from torch.utils.data import SubsetRandomSampler, DataLoader  # For sampling and DataLoader
import numpy as np
from sklearn.metrics import accuracy_score
import pandas as pd


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
n_new_classes = 2
n_epochs = 20
new_batch_size = 64
learning_rate = 0.001
n_features = 28 * 28
n_hidden = 100


In [ ]:
train_data = torchvision.datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=transforms.ToTensor(),
)

test_data = torchvision.datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=transforms.ToTensor(),
)

In [ ]:
class YourSampler(torch.utils.data.sampler.Sampler):
    def __init__(self, mask, data_source):
        self.mask = mask
        self.data_source = data_source

    def __iter__(self):
        return iter([i.item() for i in torch.nonzero(mask)])

    def __len__(self):
        return len(self.data_source)

mnist = torchvision.datasets.MNIST(root="data", download=True, transform=transforms.ToTensor())
mask = [1 if mnist[i][1] == 0 or mnist[i][1] == 1 else 0 for i in range(len(mnist))]
mask = torch.tensor(mask)
sampler = YourSampler(mask, mnist)
new_data = torch.utils.data.DataLoader(mnist,sampler = sampler, shuffle=False)



In [ ]:
all_images = []
all_labels = []

for images, labels in new_data:
    all_images.append(images)
    all_labels.append(labels)

all_images = torch.cat(all_images)
all_labels = torch.cat(all_labels)


In [ ]:
import pandas as pd

data_list = []
for batch_data, batch_labels in new_data:
    batch_data = batch_data.view(batch_data.size(0), -1).numpy()
    batch_labels = batch_labels.numpy()
    data_list.extend(zip(batch_data, batch_labels))

full_data = pd.DataFrame(data_list, columns=['x', 'y'])


In [ ]:
batch_size = 10
p = 0.6
q = 0.4

batch_index = 0

full_data['Batch'] = -1

while(True):
    prop = p if (batch_index % 2 == 0) else q
    num_class_a_elts = int(batch_size * prop)
    num_class_b_elts = batch_size - num_class_a_elts

    class_a_unassigned = full_data[(full_data['y'] == 0) & (full_data['Batch'] == -1)]
    class_b_unassigned = full_data[(full_data['y'] == 1) & (full_data['Batch'] == -1)]

    if class_a_unassigned.shape[0] < num_class_a_elts or \
        class_b_unassigned.shape[0] < num_class_b_elts:
        break
    else:
        full_data.loc[class_a_unassigned.sample(num_class_a_elts).index, 'Batch'] = batch_index
        full_data.loc[class_b_unassigned.sample(num_class_b_elts).index, 'Batch'] = batch_index
        batch_index += 1

In [ ]:
test_size = 0.3
full_data = full_data[full_data['Batch'] != -1]

num_batches = full_data['Batch'].nunique()
print("Number of batches:", num_batches)

num_test_batches = int(num_batches * test_size)
num_train_batches = num_batches - num_test_batches

training_data = full_data[full_data['Batch'] < num_train_batches]
test_data = full_data[full_data['Batch'] >= num_train_batches]

num_batches = training_data['Batch'].nunique()
print("Number of training batches:", num_batches)

num_t_batches = test_data['Batch'].nunique()
print("Number of testing batches:", num_t_batches)

input_col = ['x']



Number of batches: 1184
Number of training batches: 829
Number of testing batches: 355


In [ ]:
training_data_batches = training_data.groupby('Batch')


In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self, n_features: int, n_hidden: int, n_new_classes: int) -> None:
        super(NeuralNetwork, self).__init__()
        self.h1 = nn.Linear(n_features, n_hidden)
        self.h2 = nn.Linear(n_hidden, n_hidden)
        self.h3 = nn.Linear(n_hidden, n_hidden)
        self.out = nn.Linear(n_hidden, n_new_classes)

    def forward(self, x):
        out = torch.relu(self.h1(x))
        out = torch.relu(self.h2(out))
        out = torch.relu(self.h3(out))
        out = self.out(out)
        return out

In [ ]:
class LossFunction(nn.Module):
    def __init__(self):
        super(LossFunction, self).__init__()

    def forward(self, predictions, mean_l):
        mean_p = torch.mean(predictions)
        mean_l = torch.tensor(mean_l, dtype=torch.float32)

        return torch.square(mean_p - mean_l)

In [ ]:
device = torch.device('cpu')
model = NeuralNetwork(n_features, n_hidden, n_new_classes).to(device)

loss_fn = LossFunction()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [ ]:
from sklearn.metrics import confusion_matrix

In [ ]:
def train(epoch):

    model.train()
    loss_epoch = []

    for num, batch in training_data_batches:

        prop = batch['y'].value_counts()[0] / batch.shape[0]

        model_input = torch.tensor(batch['x'].tolist(), dtype=torch.float32)

        output = model(model_input)

        loss = loss_fn(output, prop)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        loss_epoch.append(loss.detach().numpy())

    input_data = torch.tensor(test_data['x'].tolist(), dtype=torch.float32)

    predictions = model(input_data).detach().numpy()
    predictions = np.where(predictions < 0.5, 1, 0)
    predictions = np.delete(predictions, 1, axis=1)

    targets = test_data['y']
    test_acc = accuracy_score(targets, predictions)

    # Compute confusion matrix
    cm = confusion_matrix(targets, predictions)
    print("Confusion Matrix:\n", cm)


    class_0_data = test_data[test_data['y'] == 1]
    input_data_class_0 = torch.tensor(class_0_data['x'].tolist(), dtype=torch.float32)
    predictions_class_0 = model(input_data_class_0).detach().numpy()
    mu = np.mean(predictions_class_0)

    class_1_data = test_data[test_data['y'] == 0]
    input_data_class_1 = torch.tensor(class_1_data['x'].tolist(), dtype=torch.float32)
    predictions_class_1 = model(input_data_class_1).detach().numpy()
    nu = np.mean(1 - predictions_class_1)

    return loss_epoch, test_acc, mu, nu



In [ ]:
loss_history = []
test_history = []
mu_history = []
nu_history = []

for epoch in range(1, n_epochs + 1):
    loss_epoch, test_acc, mu, nu = train(epoch)
    print("Epoch:", epoch, "Loss:", np.mean(loss_epoch), "Test Accuracy:", test_acc, "mu, nu:", mu, nu)
    loss_history.append(loss_epoch)
    test_history.append(test_acc)
    mu_history.append(mu)
    nu_history.append(nu)

Confusion Matrix:
 [[   3 1771]
 [   0 1776]]
Epoch: 1 Loss: 0.10748197 Test Accuracy: 0.5011267605633802 mu, nu: 0.36502537 0.59161603
Confusion Matrix:
 [[1712   62]
 [ 327 1449]]
Epoch: 2 Loss: 0.011988951 Test Accuracy: 0.8904225352112676 mu, nu: 0.4541743 0.47807282
Confusion Matrix:
 [[1744   30]
 [ 582 1194]]
Epoch: 3 Loss: 0.008701934 Test Accuracy: 0.8276056338028169 mu, nu: 0.4610093 0.461415
Confusion Matrix:
 [[1747   27]
 [ 475 1301]]
Epoch: 4 Loss: 0.00852681 Test Accuracy: 0.8585915492957746 mu, nu: 0.4582226 0.4564035
Confusion Matrix:
 [[1747   27]
 [ 357 1419]]
Epoch: 5 Loss: 0.008388587 Test Accuracy: 0.8918309859154929 mu, nu: 0.4544783 0.45239133
Confusion Matrix:
 [[1747   27]
 [ 247 1529]]
Epoch: 6 Loss: 0.008249488 Test Accuracy: 0.9228169014084507 mu, nu: 0.45055702 0.44838098
Confusion Matrix:
 [[1746   28]
 [ 179 1597]]
Epoch: 7 Loss: 0.00810861 Test Accuracy: 0.9416901408450704 mu, nu: 0.446519 0.44427153
Confusion Matrix:
 [[1746   28]
 [ 128 1648]]
Epoch: 